In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from src.prepare_dataset import get_dev_df  

pd.set_option('display.max_colwidth', 1000)

/home/nik/miniforge3/envs/nlp-cw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROBERTA_BASE_PATH = Path('/home/nik/ImperialWork/nlp/coursework/outputs/2026-03-01/16-50-48/out/ensemble/dev.txt')         
BEST_MODEL_PATH = Path('/home/nik/ImperialWork/nlp/DPM_binary_classification/multirun/2026-03-02/23-07-08/44/out/ensemble/dev.txt') 

ROBERTA_BASE = 'Roberta-Base Baseline'
BEST_MODEL = 'Best Model Ensemble'

In [3]:
dev_df = get_dev_df()  
y_true = dev_df['label'].astype(int).to_numpy()
texts = dev_df['text'].astype(str).to_list()

In [4]:
def read_dev_txt(path: Path) -> np.ndarray:
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines() if ln.strip() != '']
    return np.array([int(x) for x in lines], dtype=np.int64)

pred_a = read_dev_txt(ROBERTA_BASE_PATH)
pred_b = read_dev_txt(BEST_MODEL_PATH)

assert len(pred_a) == len(y_true), f'{ROBERTA_BASE} length {len(pred_a)} != dev size {len(y_true)}'
assert len(pred_b) == len(y_true), f'{BEST_MODEL} length {len(pred_b)} != dev size {len(y_true)}'

In [5]:
cmp = pd.DataFrame({
    'text': texts,
    'y_true': y_true,
    'pred_a': pred_a,
    'pred_b': pred_b,
})

cmp['a_correct'] = (cmp['pred_a'] == cmp['y_true'])
cmp['b_correct'] = (cmp['pred_b'] == cmp['y_true'])

# 4-way outcome bucket
cmp['bucket4'] = np.select([
        cmp['a_correct'] & cmp['b_correct'],
        (~cmp['a_correct']) & (~cmp['b_correct']),
        cmp['a_correct'] & (~cmp['b_correct']),
        (~cmp['a_correct']) & cmp['b_correct'],
    ],
    [
        'both_correct',
        'both_wrong',
        'A_correct_B_wrong',
        'A_wrong_B_correct',
    ],
    default='(unexpected)',
)

cmp['error_type_a'] = np.select(
    [(cmp['y_true'] == 0) & (cmp['pred_a'] == 1), (cmp['y_true'] == 1) & (cmp['pred_a'] == 0)],
    ['FP', 'FN'],
    default='OK',
)
cmp['error_type_b'] = np.select(
    [(cmp['y_true'] == 0) & (cmp['pred_b'] == 1), (cmp['y_true'] == 1) & (cmp['pred_b'] == 0)],
    ['FP', 'FN'],
    default='OK',
)

cmp['label_name'] = np.where(cmp['y_true'] == 1, 'PCL', 'nonPCL')
cmp['bucket8'] = cmp['bucket4'].astype(str) + '_' + cmp['label_name'].astype(str)

BUCKET4 = ['both_correct', 'both_wrong', 'A_correct_B_wrong', 'A_wrong_B_correct']
BUCKET8 = [f'{b}_nonPCL' for b in BUCKET4] + [f'{b}_PCL' for b in BUCKET4]

In [6]:
def sample_examples(
    df: pd.DataFrame,
    bucket8: str,
    n: int = 8,
    seed: int = 0,
    error_type_a: str | None = None,
    error_type_b: str | None = None,
) -> pd.DataFrame:
    sub = df[df['bucket8'] == bucket8]

    if error_type_a is not None:
        sub = sub[sub['error_type_a'] == error_type_a]
    if error_type_b is not None:
        sub = sub[sub['error_type_b'] == error_type_b]

    if len(sub) == 0:
        return sub

    sub = sub.sample(n=min(n, len(sub)), random_state=seed)
    cols = ['y_true', 'pred_a', 'pred_b', 'bucket4', 'bucket8', 'error_type_a', 'error_type_b', 'text']
    return sub[cols].reset_index(drop=True)


In [7]:
for b in BUCKET8:
    print(b)
    display(sample_examples(cmp, b, n=10, seed=0))

both_correct_nonPCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,0,0,0,both_correct,both_correct_nonPCL,OK,OK,""" There should be strengthening of a nutrition surveillance by proper mapping of high risk and vulnerable districts . Nutrition should be made central to the development agenda , "" the report added ."
1,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"I 've met countless women with rich , dark , gorgeous skin who steer clear of make-up , opting , instead , to invest in good mascara and shaping their eyebrows ."
2,0,0,0,both_correct,both_correct_nonPCL,OK,OK,""" I think you will see President Trump being willing to give legal status to some of the illegal immigrants who are not bad hombres if he can get better border security and more robust legal immigration , "" he said . "" I may be wrong but I think he can fix this . "" <h> Different places , different issues"
3,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"The narrow strips that separate you from your neighbour and provide access to the back yard are really a corridor , but in garden terms , they do not need to become a straight bowling alley . A little clever thinking can turn these hopeless areas into part of your usable garden ."
4,0,0,0,both_correct,both_correct_nonPCL,OK,OK,""" In addition , the indefinite detention of many vulnerable migrants , including families with small children , is cruel and inhuman . """
5,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Nine out of 10 countries ranked as most vulnerable to natural hazards in the index are in sub-Saharan Africa , and 23 of 25 are on the continent ."
6,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"Refugee Crisis The situation in Logo , Ukum , Buruku and Agatu has resulted in a huge refugee crisis in the state ."
7,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"My conclusion is that ours is a country and a world that has turned hope into a commodity and created ruthless "" hopereneurs "" ( hope entrepreneurs ) who will do anything to exploit hopelessness for their own enrichment ."
8,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"He further ruled that Interior Cabinet Secretary ( Minister ) Joseph Nkaissery and his Principal Secretary ( Assistant Minister ) , Karanja Kibicho , had acted beyond their powers in issuing the directive . The government had set aside one billion shillings for the repatriation of the refugees ."
9,0,0,0,both_correct,both_correct_nonPCL,OK,OK,"It is a classic example -- one of 65 proposals selected from 500 applications -- of the foundation 's goal of encouraging "" innovations in the areas where there is less profit opportunity but where the impact for those in need is very high . """


both_wrong_nonPCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"The backdrop to the year has been William , Kate and Harry 's mental health campaign , Heads Together , which has encouraged people to speak about their problems or be a sympathetic ear for others in need ."
1,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"But he said when the company sent him his refund , they 're going to give all of the money to a group in Columbus that helps the homeless . "" We 're just going to donate it directly to that group and let them help other homeless people , because that 's what was intended to happen , "" he said ."
2,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"Perhaps one of Lahiri 's most lasting contributions was keeping written records of the conditions of the refugees , with vivid and heart wrenching descriptions of their plight ."
3,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,""" The UN Security Council must stand up and act to support vulnerable Palestinian people at the time when they need their protection . """
4,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"The University of Technology , Jamaica through its USAID-funded Fi Wi Jamaica Project on international Human Rights Day , Thursday , December 10 , 2015 honoured ten ( 10 ) ? unsung heroes ? and one ( 1 ) institution for their transformative and enduring humanitarian work in influencing the way that Jamaican society values the most vulnerable . The awards were presented at the 2nd Annual Essence of Humanity Ubuntu Awards ceremony held at the PCJ auditorium , New Kingston ."
5,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"In his condolence message , he said , "" All our sympathies are with the poor families as we are standing in cohesion with the Egyptian government and the people in this hour of grief . """
6,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,Geeta and the other women in their Self Help Group are also proud of the fact that other women are also getting inspired by their progress story . More and more women are getting associated with Bihan with an aim to bring about positive changes in their lives and most of them are inspired by other women who have already shown the example .
7,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"Jesus begins his teaching in Matthew with the Sermon on the Mount . One group he blesses is those in need of comfort , Blessed are they who mourn , for they will be comforted ( Mt 5:4 ) ."
8,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,The Jali family in Brown 's Farm was elated as the City of Cape Town donated wheelchairs to their disabled members .
9,0,1,1,both_wrong,both_wrong_nonPCL,FP,FP,"I mention these moments to highlight and illustrate the potential that young people have to change our society . In all these moments they were able to find a way in situations that were otherwise deemed hopeless . Where other people could not have thought that there was way , young people were able to create one ."


A_correct_B_wrong_nonPCL


...

,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"The school authorities have taken the perfect procedure , not to obstruct studies . The 300 odd athletes who engage in sports are given the freedom to practice before and after school hours under the watchful eyes of the teachers who are in-charge of sports . The talented athletes who come from poor families are taken care of adequately with contributions from parents and well wishers ."
1,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"He said the victims who are currently rendered homeless can now be relieved of troubles as the 5,000 iron sheets from Mwanza had arrived , with 1,200 already distributed to victims in Bukoba Municipality ."
2,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"10:41am - Parents of children who died must get compensation , free medicine must be provided to poor families across UP : Ram Gopal Yadav"
3,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"So when journalists and observers tell us that , "" Toronto will never be the same again , "" they are not only wrong , but are extremely unhelpful . Of course Toronto will be just the same again . Not for the poor families of the deceased , not for those struggling with ghastly and perhaps life-changing injuries , but for most of the rest of us . It wo n't change , because it never does ."
4,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"It is also good news for women 's issues , which are in desperate need of a woman 's touch . As much work as the ANC has done thus far , we have a long way to go in this and in many other regards ."
5,1,0,0,both_wrong,both_wrong_PCL,FN,FN,Famously progressive San Francisco has among America 's most eco-friendly public policies and has declared itself a sanctuary to immigrants it considers persecuted by President Donald Trump 's administration .
6,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"We are alarmed to learn of your recently circulated proposals that would eviscerate the Lifeline program and leave many of the most vulnerable people in the country without access to affordable communications . As you are well aware , the Lifeline program provides a modest monthly subsidy of $9.25 to connect low-income Americans to phone and internet services . As broadband prices continue to soar , and affordability continues to suffer , adoption gaps remain . The Lifeline program has proven critical for poor families and people of color who are caught on the wrong side of the digital divide ."
7,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"In the space of a week , four beautiful young women gave up their lives voluntarily over reasons ranging from a love affair to depression to unemployment . All were educated and from reasonably good families . And these are just some of the reports that media covered . Looked at dispassionately and from a distance , none of their problems seem insurmountable . What a waste of young , beautiful lives !"
8,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"Kyle really your a pig , lol youre also very ignorant do nt like over weight women , well have u looked in the mirror recently your FAT , YOU AND YOUR BOSS SHOULD BE SACKED never to return to radio , , how dare you say go for the disabled , your more disabled than the disabled olympians , they have a genuine heart , you do nt have one ,"
9,1,0,0,both_wrong,both_wrong_PCL,FN,FN,"Winfrey 's school is an attempt to wield philanthropy and celebrity against South Africa 's social and educational crises . High-achieving students from poor families were admitted after a rigorous application process in which Winfrey was deeply involved , and she has visited regularly to counsel her girls . She held a last , late-night "" pajama party "" with the graduates Friday ."


A_correct_B_wrong_PCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,"Hundreds of thousand Africans are graduating per year . Different from 1980s and early 1990s when college outpours got immediately absorbed in the labour market , many today are jobless and hopeless ."
1,1,1,0,A_correct_B_wrong,A_correct_B_wrong_PCL,OK,FN,""" We know Uber partners with an extensive network of drivers , and Plunket nurses support some of our most vulnerable families . Working together , we can make it simple for people to help out and make a difference , "" Jarvie says ."


A_wrong_B_correct_PCL


,y_true,pred_a,pred_b,bucket4,bucket8,error_type_a,error_type_b,text
0,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,He wants more done now to help those in need .
1,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"However , this success story is not uncommon . It often happens that people from poor families have more perseverance to fight tooth an nail in business than children of rich parents who are used to get everything they want with ease . People without a strong spirit will quickly break down and drop out from the competition ."
2,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"Balu , an honest , hard-working labourer , who was injured by army shelling about 1993 leading to partial deafness , had latterly resettled in Tellipalai . While waiting to cross the KKS Rd. , he was killed by a navy vehicle with , as I learn , defective brakes , driven by a man without a heavy vehicle licence . When development fails the most vulnerable and poor , we have lost our way . It is well to remember Tagore in his essay on Nationalism : "" ... speed comes to its end , the engagement loses its meaning and the hungry heart clamours for food , till at last she comes to the lowly reaper reaping his harvest in the sun . """
3,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"Rather sad . Good set of pictures that tells a tale of survival , subsistence living , and hopeless nature of life in the tribal societies . Exploiting an unexpected geo-political bonanza is a temporary relief that is not sustainable . Education and economic development seem miles away ."
4,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,""" In addition , the Members of Parliament will visit two projects that focus on the inclusion of street children and victims of acid attacks to demonstrate their solidarity with those vulnerable groups in need of support . Finally , on April 29 the delegation will brief the journalists on the outcome of the visit at a press conference . """
5,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,TurkIt 's heartening to see that measures are being taken in Khyber Pakhtunkhwa ( KP ) to empower women and give them work opportunities . You ! takes a look ...
6,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"KOLKATA : Sourav Kaibartya , a fisherman 's son who scored 94.2% in his higher secondary examination this year got entry into NIT Durgapur for engineering course . The boy was at a loss as to who will fund his education . That is when a corporate house came into his rescue . Thirty-seven students like him from West Bengal meritorious but from poor families will now get to continue their studies with an initiative from Magma Fincorp Ltd ."
7,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"The vast majority of girls and women caught in the exploitative global sex trade are not victims of kidnapping , like the Nigerian 276 abducted by Boko Haram , but rather of poverty . Human traffickers prey on poor families who do n't have access to education and are n't aware of their basic rights . Mired in grinding poverty , parents desperately take out loans on conditions they do n't understand , pledging their children on their debts ."
8,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"In Dublin , the Church of Ireland Archbishop Dr Michael Jackson reflected on the year drawing to a close and recalled the "" horrific conflagration at the halting site in Carrickmines "" and the death of Garda Tony Golden in Omeath , as well as the "" refugees fleeing from Syria "" and other parts of the Middle East and the "" forgotten peoples of Africa "" ."
9,1,0,1,A_wrong_B_correct,A_wrong_B_correct_PCL,FN,OK,"In his final year as president , Mr S R Nathan - together with a few of his close friends - started discussing with me the idea of starting a philanthropic fund to help "" uplift "" children from poor families ."
